In [9]:
import pickle

idx_path = '/lustre/scratch126/cellgen/lotfollahi/DATASETS/tokenizer/hst_corpus_110m_32_None_None_None_None_None_shifted_log_knn_10_precomputed_loader_idx_split_train.pkl'

with open(idx_path, 'rb') as f:
    idx = pickle.load(f)

In [12]:
len(idx)

96048116

In [14]:
from datasets import load_from_disk

In [16]:
ds = load_from_disk('/lustre/scratch126/cellgen/lotfollahi/DATASETS/gold/cell-graph-tokenizer/hst_corpus_110m/hst_corpus_110m_32_None_None_None_None_None_shifted_log_knn_10_ordered_full.dataset')

In [17]:
ds

Dataset({
    features: ['rel_x_coord', 'rel_y_coord', 'cell_id', 'batch_value', 'gene_panel_value', 'assay_value', 'species_value', 'tissue_value', 'gene_tokens', 'gene_expr', 'n_nonzero_tokens', 'batch_token', 'gene_panel_token', 'assay_token', 'species_token', 'tissue_token'],
    num_rows: 112578039
})

In [13]:
from pathlib import Path

root = Path("/lustre/scratch126/cellgen/lotfollahi/DATASETS/gold/cell-graph-tokenizer")
pattern = "None_None_None_None_None_shifted_log_knn_6_7168.dataset"

# Recursively search for directories containing the pattern in their name
matching_dirs = [
    str(p) for p in root.rglob("*")
    if p.is_dir() and pattern in p.name
]

print(f"Found {len(matching_dirs)} matching directories:")
for d in matching_dirs:
    #if d == '/lustre/scratch126/cellgen/lotfollahi/DATASETS/gold/cell-graph-tokenizer/xhb1002-AT12-1b_1p/xhb1002-AT12-1b_1p_32_None_None_None_None_None_shifted_log_knn_6_7168.dataset':
    #    continue
    #elif d == '/lustre/scratch126/cellgen/lotfollahi/DATASETS/gold/cell-graph-tokenizer/xhl1015-R6-28b_1p/xhl1015-R6-28b_1p_32_None_None_None_None_None_shifted_log_knn_6_7168.dataset':
    #    continue
    #elif d == '/lustre/scratch126/cellgen/lotfollahi/DATASETS/gold/cell-graph-tokenizer/xho1012-Hrv162-2b_1p/xho1012-Hrv162-2b_1p_32_None_None_None_None_None_shifted_log_knn_6_7168.dataset':
    #    continue
    #else:
    ds = load_from_disk(d)
    if 'batch_token' not in ds.features.keys():
        print(d)
        print(ds)
        print('===')

Found 160 matching directories:
/lustre/scratch126/cellgen/lotfollahi/DATASETS/gold/cell-graph-tokenizer/xho54-1b_1p/xho54-1b_1p_32_None_None_None_None_None_shifted_log_knn_6_7168.dataset
Dataset({
    features: ['rel_x_coord', 'rel_y_coord', 'cell_id', 'batch_value', 'gene_panel_value', 'assay_value', 'species_value', 'tissue_value', 'gene_tokens', 'gene_expr', 'n_nonzero_tokens'],
    num_rows: 234984
})
===
/lustre/scratch126/cellgen/lotfollahi/DATASETS/gold/cell-graph-tokenizer/xhl1020-3-16b_1p/xhl1020-3-16b_1p_32_None_None_None_None_None_shifted_log_knn_6_7168.dataset
Dataset({
    features: ['rel_x_coord', 'rel_y_coord', 'cell_id', 'batch_value', 'gene_panel_value', 'assay_value', 'species_value', 'tissue_value', 'gene_tokens', 'gene_expr', 'n_nonzero_tokens'],
    num_rows: 2548630
})
===
/lustre/scratch126/cellgen/lotfollahi/DATASETS/gold/cell-graph-tokenizer/xhl1005-1b_1p/xhl1005-1b_1p_32_None_None_None_None_None_shifted_log_knn_6_7168.dataset
Dataset({
    features: ['rel_x_c

In [ ]:
/lustre/scratch126/cellgen/lotfollahi/DATASETS/gold/cell-graph-tokenizer/xhb1002-AT12-1b_1p/xhb1002-AT12-1b_1p_32_None_None_None_None_None_shifted_log_knn_6_7168.dataset

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import pickle
import time
import timeit
import random

import torch
from datasets import load_from_disk

from nichejepa.masks.block_masking  import BlockMaskCollator
from nichejepa.datasets.cell_datasets import init_cell_dataset
from nichejepa.datasets.dataloaders import init_dataloader_and_sampler, CustomDistributedLengthGroupedSampler

/nfs/users/nfs_s/sb75/.venvs/nichejepa/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/nfs/users/nfs_s/sb75/.venvs/nichejepa/lib/python3.10/site-packages/dask/dataframe/__init__.py:31: FutureWarning: The legacy Dask DataFrame implementation is deprecated and will be removed in a future version. Set the configuration option `dataframe.query-planning` to `True` or None to enable the new Dask Dataframe implementation and silence this warning.
  warnings.warn(
/nfs/users/nfs_s/sb75/.venvs/nichejepa/lib/python3.10/site-packages/anndata/utils.py:434: FutureWarning: Importing read_text from `anndata` is deprecated. Import anndata.io.read_text instead.
  warnings.warn(msg, FutureWarning)


In [3]:
batch_size = 124

In [4]:
# Load Huggingface Dataset
#data_path = '/lustre/scratch126/cellgen/lotfollahi/DATASETS/gold/cell-graph-tokenizer/hst_corpus_90m/hst_corpus_90m_32_None_None_None_None_None_shifted_log_knn_10.dataset'
#data_path = '/lustre/scratch126/cellgen/lotfollahi/DATASETS/gold/cell-graph-tokenizer/hst_corpus_90m/hst_corpus_90m_32_None_None_None_None_None_shifted_log_knn_10_ordered.dataset'
#data_path = '/lustre/scratch126/cellgen/lotfollahi/DATASETS/gold/cell-graph-tokenizer/hst_corpus_110m/hst_corpus_110m_32_None_None_None_None_None_shifted_log_knn_10.dataset'
data_path = '/nfs/team361/sb75/DATASETS/gold/cell-graph-tokenizer/hst_corpus_90m/hst_corpus_90m_32_None_None_None_None_None_shifted_log_knn_10.dataset'
dataset = load_from_disk(data_path)
dataset

Dataset({
    features: ['rel_x_coord', 'rel_y_coord', 'cell_id', 'batch_value', 'gene_panel_value', 'assay_value', 'species_value', 'tissue_value', 'gene_tokens', 'gene_expr', 'n_nonzero_tokens'],
    num_rows: 88626282
})

In [5]:
datasets_folder_path = '/nfs/team361/sb75/DATASETS' # '/lustre/scratch126/cellgen/team361/DATASETS'
token_dict_path = f'{datasets_folder_path}/tokenizer/token_dictionary_homo_sapiens.pkl'

In [6]:
# Load token dict
with open(token_dict_path, 'rb') as f:
    token_dict = pickle.load(f)

token_dict_reverse =  {value: key for key, value in token_dict.items()}

spv_dict = {
    k: v for k, v in token_dict.items() if k.startswith('spv_')}
spv_start_idx = min(spv_dict.values())
max_cls_tokens = sum(1 for key in token_dict if "cls" in key)
spv_idx_subtract = spv_start_idx - 2 - max_cls_tokens

In [10]:
import math
distances = [math.sqrt(dx**2 + dy**2) for dx, dy in zip(list(dataset[0]['rel_x_coord']), list(dataset[0]['rel_y_coord']))]

In [11]:
set(distances)

{0.0,
 7.5505356196114235,
 7.850037549327304,
 8.899783864244846,
 10.130013683205174,
 11.540102631089653,
 13.211170395253891,
 16.797449474712927,
 16.83457052651216,
 17.678430078472577,
 17.82182943777086}

In [91]:
token_dict['spt_batch']

106

def flatten_special_tokens(batch):
    return {
        "batch_token": [x[0] for x in batch["batch_token"]]
        "gene_panel_token": [x[0] for x in batch["gene_panel_token"]]
        "assay_token": [x[0] for x in batch["assay_token"]]
        "species_token": [x[0] for x in batch["species_token"]]
        "tissue_token": [x[0] for x in batch["tissue_token"]]
    }

dataset = dataset.map(
    flatten_special_tokens,
    batched=True,
    num_proc=6)

In [ ]:
filtered_dataset['rel_x_coord']

In [ ]:
cell_ids = list(filtered_dataset['cell_id'])

In [ ]:
set([cell_id.split('_')[1] for cell_id in cell_ids])

In [ ]:
import scanpy as sc
adata_tmp = sc.read_h5ad('/nfs/team361/sb75/DATASETS/silver/xhk1007-CV2-41b_1p/CV2-KID-0-FO-2.h5ad')

In [ ]:
adata_tmp2 = sc.read_h5ad('/nfs/team361/sb75/DATASETS/silver/xhk1007-CV5-41b_1p/CV5-KID-0-FT-4-S3.h5ad')

In [ ]:
adata_tmp.uns

In [ ]:
adata_tmp2

In [ ]:
adata_tmp2.uns

In [ ]:
item = dataset[0]

In [9]:
columns = list(dataset.features.keys())
columns.remove("cell_id")
dataset.set_format(
    type=None, #"torch",
    columns=columns,
    output_all_columns=True)
dataset

Dataset({
    features: ['rel_x_coord', 'rel_y_coord', 'cell_id', 'batch_value', 'gene_panel_value', 'assay_value', 'species_value', 'tissue_value', 'gene_tokens', 'gene_expr', 'n_nonzero_tokens'],
    num_rows: 112578039
})

In [12]:
# Define params for Cell Dataset
token_dict_folder_path = '/lustre/scratch126/cellgen/lotfollahi/DATASETS/tokenizer/token_dictionary_homo_sapiens.pkl'

with open(token_dict_folder_path, 'rb') as file:
    token_dict = pickle.load(file)
vocab_size = len(token_dict)

seq_len_cell = 256
seq_len_neighborhood = 2560
tokenizer_type = 'cell_graph'
gt_type = 'combined'
cell_pos_enc = 'coord'
special_tokens = []
sampling_strategy = None
sep_gene_tokens_neb = False

#with open('/lustre/scratch126/cellgen/lotfollahi/DATASETS/tokenizer/hst_corpus_110m_32_None_None_None_None_None_shifted_log_knn_10_n_nonzero_tokens_train.pkl', "rb") as f: 
with open('/lustre/scratch126/cellgen/lotfollahi/DATASETS/tokenizer/hst_corpus_90m_32_None_None_None_None_None_shifted_log_knn_10_n_nonzero_tokens_train.pkl', "rb") as f: 
    n_nonzero_tokens = pickle.load(f)

In [13]:
#with open('/lustre/scratch126/cellgen/lotfollahi/DATASETS/tokenizer/hst_corpus_110m_32_None_None_None_None_None_shifted_log_knn_10_precomputed_split_train.pkl', "rb") as f: 
with open('/lustre/scratch126/cellgen/lotfollahi/DATASETS/tokenizer/hst_corpus_90m_32_None_None_None_None_None_shifted_log_knn_10_precomputed_split_train.pkl', "rb") as f: 
    train_idx = pickle.load(f)

In [10]:
train_dataset = dataset.select(train_idx)
#train_dataset = dataset

In [11]:
train_dataset

Dataset({
    features: ['rel_x_coord', 'rel_y_coord', 'cell_id', 'batch_value', 'gene_panel_value', 'assay_value', 'species_value', 'tissue_value', 'gene_tokens', 'gene_expr', 'n_nonzero_tokens'],
    num_rows: 81361647
})

In [16]:
#n_nonzero_tokens_correct = list(train_dataset['n_nonzero_tokens'])

In [ ]:
n_nonzero_tokens[:100]

In [17]:
columns = list(train_dataset.features.keys())
columns.remove("cell_id")
train_dataset.set_format(
    type="torch", # "torch",
    columns=columns,
    output_all_columns=True)

In [22]:
def run():
    for i in range(batch_size):
        data = train_dataset[i]

# time 10 runs, each repeating the loop once
elapsed = timeit.timeit(run, number=10)
print(f"Average per run: {elapsed / 10:.6f} seconds")
# Average per run: 0.070871 seconds
# Average per run: 0.620088 seconds

Average per run: 0.104441 seconds


In [23]:
random_range = random.sample(range(0, 81361647), batch_size)

def run():
    for i in random_range:
        data = train_dataset[i]

# time 10 runs, each repeating the loop once
elapsed = timeit.timeit(run, number=10)
print(f"Average per run: {elapsed / 10:.6f} seconds")
# Average per run: 3.405504 seconds
# Average per run: 4.063928 seconds

Average per run: 4.591539 seconds


In [24]:
def run():
    data = train_dataset[list(range(batch_size))]

# time 10 runs, each repeating the loop once
elapsed = timeit.timeit(run, number=10)
print(f"Average per run: {elapsed / 10:.6f} seconds")
# Average per run: 0.197345 seconds

Average per run: 0.012290 seconds


In [25]:
random_range = random.sample(range(0, 81361647), batch_size)

def run():
    data = train_dataset[random_range]

# time 10 runs, each repeating the loop once
elapsed = timeit.timeit(run, number=10)
print(f"Average per run: {elapsed / 10:.6f} seconds")
# Average per run: 5.533628 seconds

Average per run: 6.245000 seconds


In [32]:
# Initialize Cell Dataset
cell_dataset = init_cell_dataset(
    dataset=train_dataset,
    vocab_size=vocab_size,
    seq_len_cell=seq_len_cell,
    seq_len_neighborhood=seq_len_neighborhood,
    tokenizer_type=tokenizer_type,
    gt_type=gt_type,
    cell_pos_enc=cell_pos_enc,
    special_tokens=special_tokens,
    sampling_strategy=sampling_strategy,
    n_nonzero_tokens_list=n_nonzero_tokens,
    include_cell_id=False,
    sep_gene_tokens_neb=sep_gene_tokens_neb)

In [33]:
len(cell_dataset)

81361647

In [34]:
cell_data = cell_dataset[0]

tensor([ 1.,  1.,  1.,  1.,  1.,  1.,  1.,  1.,  1.,  1.,  1.,  1.,  1.,  1.,
         1.,  1.,  1.,  1.,  1.,  1.,  1.,  1.,  1.,  1.,  1.,  1.,  1.,  1.,
         1.,  1.,  1.,  1.,  1.,  1.,  1.,  1.,  1.,  1.,  1.,  1.,  1.,  1.,
         1.,  1.,  1.,  1.,  1.,  1.,  1.,  1.,  1.,  1.,  1.,  1.,  1.,  1.,
         1.,  1.,  1.,  1.,  1.,  1.,  1.,  1.,  1.,  1.,  1.,  1.,  1.,  1.,
         1.,  1.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,
         0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,
         0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,
         0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,
         0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,
         0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,
         0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,
         0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  

In [28]:
len(cell_data['tokens'])

2816

In [28]:
batch_size = 144

In [29]:
import timeit

def run():
    for i in range(batch_size):
        cell_data = cell_dataset[i]

# time 10 runs, each repeating the loop once
elapsed = timeit.timeit(run, number=10)
print(f"Average per run: {elapsed / 10:.6f} seconds")
# Average per run: 0.468819 seconds

Average per run: 0.466732 seconds


In [30]:
random_range = random.sample(range(0, 81361647), batch_size)

import timeit

def run():
    for i in random_range:
        cell_data = cell_dataset[i]

# time 10 runs, each repeating the loop once
elapsed = timeit.timeit(run, number=10)
print(f"Average per run: {elapsed / 10:.6f} seconds")
# Average per run: 4.059347 seconds

Average per run: 5.472791 seconds


In [24]:
cell_data['segments'][:100]

tensor([1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 1., 1., 1., 1., 1., 1., 1.])

In [32]:
cell_data['positions'][:513]

tensor([  1,   2,   3,   4,   5,   6,   7,   8,   9,  10,  11,  12,  13,  14,
         15,  16,  17,  18,  19,  20,  21,  22,  23,  24,  25,  26,  27,  28,
         29,  30,  31,  32,  33,  34,  35,  36,  37,  38,  39,  40,  41,  42,
         43,  44,  45,  46,  47,  48,  49,  50,  51,  52,  53,  54,  55,  56,
         57,  58,  59,  60,  61,  62,  63,  64,  65,  66,  67,  68,  69,  70,
         71,  72,  73,  74,  75,  76,  77,  78,  79,  80,  81,  82,  83,  84,
         85,  86,  87,  88,  89,  90,  91,  92,  93,  94,  95,  96,  97,  98,
         99, 100, 101, 102,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
          0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
          0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
          0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
          0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
          0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,  

In [33]:
cell_data['rel_x_coords'][:513]

tensor([-7.3698, -7.3698, -7.3698, -7.3698, -7.3698, -7.3698, -7.3698, -7.3698,
        -7.3698, -7.3698, -7.3698, -7.3698, -7.3698, -7.3698, -7.3698, -7.3698,
        -7.3698, -7.3698, -7.3698, -7.3698, -7.3698, -7.3698, -7.3698, -7.3698,
        -7.3698, -7.3698, -7.3698, -7.3698, -7.3698, -7.3698, -7.3698, -7.3698,
        -7.3698, -7.3698, -7.3698, -7.3698, -7.3698, -7.3698, -7.3698, -7.3698,
        -7.3698, -7.3698, -7.3698, -7.3698, -7.3698, -7.3698, -7.3698, -7.3698,
        -7.3698, -7.3698, -7.3698, -7.3698, -7.3698, -7.3698, -7.3698, -7.3698,
        -7.3698, -7.3698, -7.3698, -7.3698, -7.3698, -7.3698, -7.3698, -7.3698,
        -7.3698, -7.3698, -7.3698, -7.3698, -7.3698, -7.3698, -7.3698, -7.3698,
        -7.3698, -7.3698, -7.3698, -7.3698, -7.3698, -7.3698, -7.3698, -7.3698,
        -7.3698, -7.3698, -7.3698, -7.3698, -7.3698, -7.3698, -7.3698, -7.3698,
        -7.3698, -7.3698, -7.3698, -7.3698, -7.3698, -7.3698, -7.3698, -7.3698,
        -7.3698, -7.3698, -7.3698, -7.36

In [34]:
cell_data['segments'][:513]

tensor([1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 

In [ ]:
random_range_restricted = sorted(random.sample(range(0, 10000000), batch_size))

In [ ]:
import timeit

def run():
    for i in random_range_restricted:
        cell_data = cell_dataset[i]

# time 10 runs, each repeating the loop once
elapsed = timeit.timeit(run, number=10)
print(f"Average per run: {elapsed / 10:.6f} seconds") # 0,421 seconds

In [ ]:
random_range_restricted_2 = sorted(random.sample(range(0, 10000), batch_size))

In [ ]:
import timeit

def run():
    for i in random_range_restricted_2:
        cell_data = cell_dataset[i]

# time 10 runs, each repeating the loop once
elapsed = timeit.timeit(run, number=10)
print(f"Average per run: {elapsed / 10:.6f} seconds") # 0,421 seconds

In [ ]:
cell_dataset[1000]['rel_x_coords'][-500:]

In [ ]:
cell_dataset[1000]['rel_y_coords'][-500:]

In [ ]:
import timeit

def run():
    for i in range(batch_size):
        cell_data = cell_dataset[81361646 - i]

# time 10 runs, each repeating the loop once
elapsed = timeit.timeit(run, number=10)
print(f"Average per run: {elapsed / 10:.6f} seconds")
# Average per run: 0.468819 seconds

In [35]:
n_targets = 1
n_contexts = 1
n_segments = 11
n_special_tokens = 0
per_block_mask_ratio = 0.6
sample_segments = True

In [36]:
mask_collator = BlockMaskCollator(
    n_targets=n_targets,
    n_contexts=n_contexts,
    n_segments=n_segments,
    seq_len_cell=seq_len_cell,
    seq_len_neighborhood=seq_len_neighborhood,
    n_special_tokens=n_special_tokens,
    per_block_mask_ratio=per_block_mask_ratio,
    sample_segments=sample_segments,
    sample_gene_masks=True)

In [37]:
world_size = 1
rank = 0
pin_memory = True
num_workers = 0

In [38]:
distributed = True # True, False

if not distributed:
    loader, sampler = init_dataloader_and_sampler(
        cell_dataset=cell_dataset,
        batch_size=batch_size,
        distributed=distributed,
        world_size=world_size,
        rank=rank,
        collate_fn=mask_collator,
        pin_memory=pin_memory,
        num_workers=num_workers,
        drop_last=True,
        #prefetch_factor=4,
        persistent_workers=False)
else:
    loader, sampler = init_dataloader_and_sampler(
        cell_dataset=cell_dataset,
        batch_size=batch_size,
        distributed=distributed,
        world_size=world_size,
        rank=rank,
        collate_fn=mask_collator,
        pin_memory=pin_memory,
        num_workers=num_workers,
        drop_last=True,
        #prefetch_factor=4,
        persistent_workers=False)

In [43]:
#device = 'cuda'
device = 'cpu'

In [44]:
for itr, (udata, masks_enc, masks_pred, masks_attention) in enumerate(loader):
    if itr == 0:
        start = time.time()
    elapsed = time.time() - start
    print(f"Time: {elapsed:.6f} seconds")
    if itr == 3:
        break

Time: 0.000001 seconds
Time: 2.682327 seconds
Time: 3.132770 seconds
Time: 3.589140 seconds


In [45]:
print("Iter with 32 GPUS:", int(len(loader) / 32 / 16))

Iter with 32 GPUS: 1302


In [ ]:
for itr, (udata, masks_enc, masks_pred, masks_attention) in enumerate(loader):
    if itr == 0:
        start = time.time()
    elapsed = time.time() - start
    print(f"Time: {elapsed:.6f} seconds")
    if itr == 3:
        break

In [46]:
import timeit
import torch, time

def run_one_iter():
    for itr, (udata, masks_enc, masks_pred, masks_attention) in enumerate(loader):
        for key, val in udata.items():
            udata[key] = val.to(device, non_blocking=True)
        masks_enc = [u.to(device, non_blocking=True) for u in masks_enc]
        masks_pred = [u.to(device, non_blocking=True) for u in masks_pred]
        masks_attention = masks_attention.to(device, non_blocking=True)
        break
    
    if device == "cuda":
        torch.cuda.synchronize()

# --- First iteration timing ---
start = time.time()
run_one_iter()
if device == "cuda":
    torch.cuda.synchronize()
first_elapsed = time.time() - start
print(f"First iteration: {first_elapsed:.6f} seconds")

# --- Average over next 10 iterations ---
elapsed = timeit.timeit(run_one_iter, number=10)
print(f"Average over next 10: {elapsed / 10:.6f} seconds")

First iteration: 90.887117 seconds


Exception ignored in: <bound method IPythonKernel._clean_thread_parent_frames of <ipykernel.ipkernel.IPythonKernel object at 0x14d138845690>>
Traceback (most recent call last):
  File "/nfs/users/nfs_s/sb75/.venvs/nichejepa/lib/python3.10/site-packages/ipykernel/ipkernel.py", line 775, in _clean_thread_parent_frames
    def _clean_thread_parent_frames(
KeyboardInterrupt: 
Exception ignored in: <bound method IPythonKernel._clean_thread_parent_frames of <ipykernel.ipkernel.IPythonKernel object at 0x14d138845690>>
Traceback (most recent call last):
  File "/nfs/users/nfs_s/sb75/.venvs/nichejepa/lib/python3.10/site-packages/ipykernel/ipkernel.py", line 775, in _clean_thread_parent_frames
    def _clean_thread_parent_frames(
KeyboardInterrupt: 


KeyboardInterrupt: 

In [47]:
for itr, (udata, masks_enc, masks_pred, masks_attention) in enumerate(loader):
    for key, val in udata.items():
        udata[key] = val.to(device, non_blocking=True)
    masks_enc = [u.to(device, non_blocking=True) for u in masks_enc]
    masks_pred = [u.to(device, non_blocking=True) for u in masks_pred]
    masks_attention = masks_attention.to(device, non_blocking=True)
    break

In [69]:
udata

{'tokens': tensor([[ 3357,  4496, 13680,  ...,     0,     0,     0],
         [18775,  4496, 18568,  ...,     0,     0,     0],
         [18775, 16259,  6890,  ...,     0,     0,     0],
         ...,
         [18775,  7857,  7700,  ...,     0,     0,     0],
         [18775, 16259,   813,  ...,     0,     0,     0],
         [ 6967,  9440,  9017,  ...,     0,     0,     0]]),
 'values': tensor([[3.1355, 2.8332, 2.3979,  ..., 0.0000, 0.0000, 0.0000],
         [3.3322, 2.4849, 2.1972,  ..., 0.0000, 0.0000, 0.0000],
         [2.7081, 2.3979, 2.1972,  ..., 0.0000, 0.0000, 0.0000],
         ...,
         [2.7081, 2.5649, 2.3026,  ..., 0.0000, 0.0000, 0.0000],
         [3.6109, 2.9444, 2.6391,  ..., 0.0000, 0.0000, 0.0000],
         [2.3979, 2.3026, 1.9459,  ..., 0.0000, 0.0000, 0.0000]]),
 'rel_x_coords': tensor([[-2.7588, -2.7588, -2.7588,  ...,    -inf,    -inf,    -inf],
         [ 0.0449,  0.0449,  0.0449,  ...,    -inf,    -inf,    -inf],
         [ 6.5972,  6.5972,  6.5972,  ...,    

In [54]:
(udata['tokens'][0] != 0).sum()

tensor(768)

In [70]:
(udata['values'][0] != 0).sum()

tensor(768)

In [71]:
(udata['rel_x_coords'][0] != 0).sum()

tensor(2816)

In [77]:
import math
distances = [math.sqrt(dx**2 + dy**2) for dx, dy in zip(udata['rel_x_coords'][0], udata['rel_y_coords'][0])]

In [86]:
udata['segments'][0][513:1800]

tensor([3., 3., 3.,  ..., 0., 0., 0.])

In [80]:
udata['rel_x_coords'][0][:513]

tensor([ -2.7588,  -2.7588,  -2.7588,  -2.7588,  -2.7588,  -2.7588,  -2.7588,
         -2.7588,  -2.7588,  -2.7588,  -2.7588,  -2.7588,  -2.7588,  -2.7588,
         -2.7588,  -2.7588,  -2.7588,  -2.7588,  -2.7588,  -2.7588,  -2.7588,
         -2.7588,  -2.7588,  -2.7588,  -2.7588,  -2.7588,  -2.7588,  -2.7588,
         -2.7588,  -2.7588,  -2.7588,  -2.7588,  -2.7588,  -2.7588,  -2.7588,
         -2.7588,  -2.7588,  -2.7588,  -2.7588,  -2.7588,  -2.7588,  -2.7588,
         -2.7588,  -2.7588,  -2.7588,  -2.7588,  -2.7588,  -2.7588,  -2.7588,
         -2.7588,  -2.7588,  -2.7588,  -2.7588,  -2.7588,  -2.7588,  -2.7588,
         -2.7588,  -2.7588,  -2.7588,  -2.7588,  -2.7588,  -2.7588,  -2.7588,
         -2.7588,  -2.7588,  -2.7588,  -2.7588,  -2.7588,  -2.7588,  -2.7588,
         -2.7588,  -2.7588,  -2.7588,  -2.7588,  -2.7588,  -2.7588,  -2.7588,
         -2.7588,  -2.7588,  -2.7588,  -2.7588,  -2.7588,  -2.7588,  -2.7588,
         -2.7588,  -2.7588,  -2.7588,  -2.7588,  -2.7588,  -2.75

In [78]:
distances

[8.860825085250598,
 8.860825085250598,
 8.860825085250598,
 8.860825085250598,
 8.860825085250598,
 8.860825085250598,
 8.860825085250598,
 8.860825085250598,
 8.860825085250598,
 8.860825085250598,
 8.860825085250598,
 8.860825085250598,
 8.860825085250598,
 8.860825085250598,
 8.860825085250598,
 8.860825085250598,
 8.860825085250598,
 8.860825085250598,
 8.860825085250598,
 8.860825085250598,
 8.860825085250598,
 8.860825085250598,
 8.860825085250598,
 8.860825085250598,
 8.860825085250598,
 8.860825085250598,
 8.860825085250598,
 8.860825085250598,
 8.860825085250598,
 8.860825085250598,
 8.860825085250598,
 8.860825085250598,
 8.860825085250598,
 8.860825085250598,
 8.860825085250598,
 8.860825085250598,
 8.860825085250598,
 8.860825085250598,
 8.860825085250598,
 8.860825085250598,
 8.860825085250598,
 8.860825085250598,
 8.860825085250598,
 8.860825085250598,
 8.860825085250598,
 8.860825085250598,
 8.860825085250598,
 8.860825085250598,
 8.860825085250598,
 8.860825085250598,


In [72]:
(udata['positions'][0] != 0).sum()

tensor(768)

In [73]:
(udata['segments'][0] != 0).sum()

tensor(768)

In [55]:
(udata['tokens'][1] != 0).sum()

tensor(768)

In [56]:
(udata['tokens'][2] != 0).sum()

tensor(768)

In [61]:
(udata['tokens'][30] != 0).sum()

tensor(768)

In [62]:
udata['tokens'].shape

torch.Size([144, 2816])

In [68]:
masks_attention[2].sum()

tensor(768)

In [ ]:
idx_temp = [79009928, 61334070, 61012720, 59885839, 51564208, 51071983, 50578755, 51490262, 21762874, 50991337, 20596439, 49076277, 55395062, 50791923, 51597330]

In [ ]:
dataset_tmp = train_dataset.select(idx_temp)

In [ ]:
len(sampler.lengths)

In [ ]:
dataset_tmp['n_nonzero_tokens']

In [ ]:
g = torch.Generator()
g.manual_seed(0)
indices = sampler._get_length_grouped_indices(generator=g)

In [ ]:
indices[:100]

In [ ]:
train_dataset

In [ ]:
ordered_dataset = train_dataset.select(indices[-100:])

In [ ]:
train_dataset[79009928]['n_nonzero_tokens']

In [ ]:
list(ordered_dataset['n_nonzero_tokens'])

In [ ]:
print(torch.count_nonzero(cell_dataset[79009928]['tokens']))
print(torch.count_nonzero(cell_dataset[61334070]['tokens']))
print(torch.count_nonzero(cell_dataset[61012720]['tokens']))
print(torch.count_nonzero(cell_dataset[59885839]['tokens']))
print(torch.count_nonzero(cell_dataset[51564208]['tokens']))
print(torch.count_nonzero(cell_dataset[51071983]['tokens']))
print(torch.count_nonzero(cell_dataset[50578755]['tokens']))
print(torch.count_nonzero(cell_dataset[51490262]['tokens']))

In [ ]:
with open('/lustre/scratch126/cellgen/lotfollahi/DATASETS/tokenizer/hst_corpus_90m_32_None_None_None_None_None_shifted_log_knn_10_precomputed_loader_idx.pkl', "wb") as f: 
    pickle.dump(indices, f)

In [ ]:
with open('/lustre/scratch126/cellgen/lotfollahi/DATASETS/tokenizer/hst_corpus_90m_32_None_None_None_None_None_shifted_log_knn_10_precomputed_loader_idx.pkl', "rb") as f: 
    indices2 = pickle.load(f)

In [ ]:
indices[:100]

In [ ]:
indices2[:100]

In [ ]:
test = torch.tensor([0, 0, 0, 1, 1, 1, 2, 2, 2])

In [ ]:
mask = test == 1

In [ ]:
mask

In [ ]:
torch.nonzero(mask, as_tuple=True)[0]

In [ ]:
import numpy as np
import scanpy as sc

from nichejepa.preprocessors.graph import construct_neighbor_graph

In [ ]:
adata_neigh = sc.read_h5ad('/nfs/team361/sb75/DATASETS/silver/xhe1008-1-20b_1p/section020_SOB26.h5ad')

In [ ]:
n_neighs = 10
radius = None
delaunay = None

In [ ]:
print('Computing spatial neighborhood...')
# Construct neighbor graph
adata_neigh = construct_neighbor_graph(
    adata_neigh,
    n_neighs=n_neighs,
    radius=radius,
    delaunay=delaunay,
    include_self_loop=False,
    compute_neighbor_counts=False)

In [ ]:
adata_neigh.obsp['spatial_connectivities'][0].indices

In [ ]:
adata_neigh.obsp['spatial_connectivities'][0].nonzero()[1]

In [ ]:
adata_neigh.obsp['spatial_distances'].indptr[2]

In [ ]:
adata_neigh.obsp['spatial_distances'].indptr[3]

In [ ]:
cell_start = adata_neigh.obsp['spatial_distances'].indptr[0]
cell_end = adata_neigh.obsp['spatial_distances'].indptr[0+1]
cell_distances = adata_neigh.obsp[
    'spatial_distances'].data[cell_start:cell_end]
cell_distances

In [ ]:
rel_x_coord = []
rel_y_coord = []

neighbor_indices = adata_neigh.obsp[
    'spatial_connectivities'][0].indices
for j in neighbor_indices:
    rel_x_coord.extend([(
        adata_neigh.obsm['spatial'][j, 0] - adata_neigh.obsm['spatial'][0, 0]
        ).tolist()])
    rel_y_coord.extend([(
        adata_neigh.obsm['spatial'][j, 1] - adata_neigh.obsm['spatial'][0, 1]
        ).tolist()])

print(rel_x_coord)
print(rel_y_coord)

In [ ]:
-7-(-10)

In [ ]:
np.sqrt(5.62371826171875**2 + 10.005859375 **2)

In [ ]:
np.sqrt(rel_x_coord**2 + rel_y_coord**2)

In [ ]:
adata_neigh